In [33]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
dataset = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", dataset.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [34]:
import torch

In [35]:
X = dataset["review"]
y = dataset["sentiment"]

In [36]:
y.unique()
y = y.map({"positive" : 1, "negative" : 0})

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

In [38]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [9]:
X_train

,review
39087,That's what I kept asking myself during the ma...
30893,I did not watch the entire movie. I could not ...
45278,A touching love story reminiscent of In the M...
16398,This latter-day Fulci schlocker is a totally a...
13653,"First of all, I firmly believe that Norwegian ..."
...,...
11284,`Shadow Magic' recaptures the joy and amazemen...
44732,I found this movie to be quite enjoyable and f...
38158,Avoid this one! It is a terrible movie. So wha...
860,This production was quite a surprise for me. I...


In [39]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [40]:
train_encoding = tokenizer(
    X_train.tolist(),
    padding=True,
    truncation=True,
    max_length=256
)

test_encoding = tokenizer(
    X_test.tolist(),
    padding=True,
    truncation=True,
    max_length=256
)

In [14]:
train_encoding = tokenize_reviews(X_train)
test_encoding = tokenize_reviews(X_test)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2357: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [17]:
train_encoding[3]

[101,
 2023,
 3732,
 1011,
 2154,
 11865,
 15472,
 2072,
 8040,
 7317,
 7432,
 2121,
 2003,
 1037,
 6135,
 11113,
 7274,
 9067,
 9530,
 3597,
 7542,
 7149,
 2007,
 2019,
 4297,
 23086,
 18503,
 2099,
 1006,
 12049,
 11085,
 7952,
 1007,
 2040,
 7288,
 2630,
 4783,
 4232,
 1011,
 2806,
 2000,
 3477,
 2125,
 2010,
 2412,
 1011,
 4803,
 13930,
 2011,
 7367,
 8566,
 6129,
 2070,
 1997,
 1996,
 1057,
 25394,
 4355,
 7743,
 2229,
 2017,
 2097,
 2412,
 3913,
 2115,
 2159,
 2006,
 1998,
 2040,
 2074,
 4148,
 2000,
 2022,
 7272,
 24835,
 999,
 1996,
 11865,
 15472,
 2072,
 1011,
 17430,
 5896,
 2036,
 9530,
 18886,
 6961,
 2000,
 13265,
 1037,
 2261,
 2304,
 2135,
 21699,
 3787,
 1011,
 2029,
 2069,
 2765,
 1999,
 2070,
 4895,
 11263,
 10695,
 2100,
 2449,
 5994,
 1037,
 11547,
 2029,
 2180,
 1005,
 1056,
 2994,
 2404,
 1010,
 2019,
 3850,
 3220,
 6778,
 2040,
 2180,
 1005,
 1056,
 2644,
 4823,
 1010,
 4385,
 1012,
 1011,
 2025,
 2000,
 5254,
 1037,
 2079,
 27877,
 24930,
 2121,
 4323,
 3442,
 

In [18]:
from torch.utils.data import Dataset, DataLoader

In [21]:
class CustomDataset(Dataset):

  def __init__(self, encoding, label):

    self.encoding = encoding
    self.label = label

  def __len__(self):

    return len(self.label)

  def __getitem__(self, idx):

    item = {}

    for key, value in self.encoding.items():
      item[key] = torch.tensor(value[idx])

    item["label"] = torch.tensor(self.label[idx])

    return item

In [22]:
train_dataset = CustomDataset(train_encoding, y_train.tolist())
test_dataset = CustomDataset(test_encoding, y_test.tolist())

In [23]:
train_dataset

In [30]:
# Loader

train_loader = DataLoader(train_dataset, batch_size = 100, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 100)

In [26]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [31]:
model.train()

epochs = 3

for epoch in range(epochs):

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss/len(train_loader):.4f}")

AttributeError: 'list' object has no attribute 'items'

In [32]:
print(type(train_encoding))
print(type(test_encoding))
print(type(train_encoding[0]))


<class 'list'>
<class 'list'>
<class 'list'>
